# 🔍 Sinhala OCR: LightOnOCR-2-1B Base Accuracy Benchmark

This notebook evaluates the **zero-shot / out-of-the-box base accuracy** of **LightOnOCR-2-1B** on Sinhala text.

### 🎯 Objectives
1. **Baseline Measurement**: Determine how well the pre-trained LightOnOCR model understands Sinhala script without any fine-tuning.
2. **Metric Evaluation**: Calculate standard OCR performance metrics:
   - **Character Error Rate (CER)**
   - **Word Error Rate (WER)**
   - **Exact Match Sequence Accuracy (%)
   - **Inference Latency** (per sample & total)
3. **Dataset**: Benchmark on [`Ransaka/sinhala_synthetic_ocr-large`](https://huggingface.co/datasets/Ransaka/sinhala_synthetic_ocr-large), which covers 5 standard Sinhala font families (*Noto Sans Sinhala, Gemunu Libre, Noto Serif Sinhala, Yaldevi, Abhaya Libre*).
4. **Error Analysis**: Inspect best-performing and worst-performing examples to identify failure modes (character confusions, missing diacritics, script hallucination).
5. **Export**: Save detailed per-sample predictions to CSV for comparison against subsequent fine-tuned models.

## 0. Installation & Dependencies
Uncomment and run the cell below if running in Google Colab or an environment without the required packages.

In [ ]:
# Install only the packages not pre-installed in Google Colab
# (Colab already includes torch, torchvision, pandas, numpy, pillow, matplotlib, tqdm)
# Keeping Colab's native pandas untouched avoids C-extension conflicts in Python 3.13!
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q datasets jiwer python-Levenshtein accelerate


## 1. Imports & System Configuration
Set up environment, hardware detection, reproducibility seeds, and logging.

In [ ]:
import os
import time
import random
import unicodedata
from typing import List, Dict, Any, Tuple

import torch
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import jiwer
from datasets import load_dataset

# Hugging Face model and processor loaders
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    try:
        from transformers import AutoModelForVision2Seq as AutoModelForImageTextToText
    except ImportError:
        from transformers import AutoModel as AutoModelForImageTextToText

# Set random seeds for deterministic evaluation
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Detect computing device and optimal precision
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    device_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ Using CUDA: {device_name} ({gpu_mem:.2f} GB VRAM)")
    print(f"✅ Computation Dtype: {DTYPE}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float32
    print("✅ Using Apple MPS (float32)")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("⚠️ CUDA not available. Running on CPU (float32).")


## 2. Benchmark Configuration
Configure model variant, dataset split, evaluation sample size, and output destination.

In [ ]:
class BenchmarkConfig:
    """Configuration parameters for the LightOnOCR base accuracy benchmark."""
    # Model identifier on Hugging Face:
    # Options: 'lightonai/LightOnOCR-2-1B' (instruction/OCR tuned) or 'lightonai/LightOnOCR-2-1B-base' (pre-trained base)
    model_id: str = "lightonai/LightOnOCR-2-1B"
    
    # Benchmark dataset
    dataset_name: str = "Ransaka/sinhala_synthetic_ocr-large"
    dataset_split: str = "train"
    
    # Evaluation sample size:
    # Set to an integer (e.g. 100 or 200) for a fast evaluation, or None to evaluate the complete split (6,969 samples)
    num_eval_samples: int | None = 100
    
    # Generation parameters
    max_new_tokens: int = 512
    
    # Results output
    results_csv_path: str = "lightonocr_base_accuracy_results.csv"
    summary_json_path: str = "lightonocr_base_accuracy_summary.json"

config = BenchmarkConfig()
print(f"Configured Model: {config.model_id}")
print(f"Configured Dataset: {config.dataset_name} (split: {config.dataset_split})")
print(f"Evaluation Samples: {config.num_eval_samples if config.num_eval_samples else 'All available samples'}")

## 3. Load and Prepare the Benchmark Dataset
We download/stream `Ransaka/sinhala_synthetic_ocr-large` from Hugging Face and prepare an evaluation subset.

In [ ]:
print(f"Loading dataset '{config.dataset_name}'...")
raw_dataset = load_dataset(config.dataset_name, split=config.dataset_split)
total_available = len(raw_dataset)
print(f"✅ Successfully loaded {total_available:,} total samples.")
print(f"Columns: {raw_dataset.column_names}")

# Detect image and label column names
image_col = next((c for c in ["image", "img"] if c in raw_dataset.column_names), None)
text_col = next((c for c in ["text", "label", "ground_truth", "transcription"] if c in raw_dataset.column_names), None)

if not image_col or not text_col:
    raise ValueError(f"Could not detect image/text columns. Found columns: {raw_dataset.column_names}")

print(f"Detected Image Column: '{image_col}', Text Column: '{text_col}'")

# Subsample evaluation set deterministically if specified
if config.num_eval_samples is not None and config.num_eval_samples < total_available:
    eval_dataset = raw_dataset.shuffle(seed=SEED).select(range(config.num_eval_samples))
    print(f"✅ Selected evaluation subset of {len(eval_dataset)} samples (Seed: {SEED}).")
else:
    eval_dataset = raw_dataset
    print(f"✅ Evaluating full dataset of {len(eval_dataset)} samples.")

### Preview Benchmark Samples
Let's visually inspect four random samples from the dataset alongside their Sinhala ground truth text.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 6))
sample_indices = random.sample(range(len(eval_dataset)), min(4, len(eval_dataset)))

for ax, idx in zip(axes.flat, sample_indices):
    item = eval_dataset[idx]
    img = item[image_col].convert("RGB")
    gt = item[text_col]
    ax.imshow(img)
    ax.set_title(f"Sample #{idx}\nGround Truth: {gt[:45]}..." if len(gt) > 45 else f"Sample #{idx}\nGT: {gt}", fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Load LightOnOCR Model & Processor
Load the model checkpoint into memory with appropriate precision.

In [ ]:
print(f"Loading model and processor for '{config.model_id}'...")
start_time = time.time()
processor = AutoProcessor.from_pretrained(config.model_id, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    config.model_id,
    torch_dtype=DTYPE,
    trust_remote_code=True
).to(DEVICE)

model.eval()
load_duration = time.time() - start_time

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded in {load_duration:.2f}s")
print(f"✅ Total parameters: {total_params / 1e9:.2f}B (Trainable: {trainable_params:,})")
print(f"✅ Model device: {model.device}, Dtype: {next(model.parameters()).dtype}")

## 5. Inference Pipeline & Smoke Test
The `extract_sinhala_text` function converts an image into the chat conversation format expected by LightOnOCR, applies the chat template to produce model inputs, generates tokens with greedy decoding, and decodes the result.

In [ ]:
def extract_sinhala_text(
    image: Image.Image,
    model: torch.nn.Module,
    processor: Any,
    device: str,
    dtype: torch.dtype,
    max_new_tokens: int = 512
) -> Tuple[str, float]:
    """
    Perform OCR text extraction on a single image using LightOnOCR.
    
    Returns:
        Tuple of (predicted_text: str, latency_seconds: float)
    """
    # Ensure RGB format
    if image.mode != "RGB":
        image = image.convert("RGB")
        
    # Format conversation as required by LightOnOCR
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image}
            ]
        }
    ]
    
    start_t = time.perf_counter()
    
    # Process image with chat template
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )
    
    # Move inputs to target device and cast floating point tensors
    model_inputs = {
        k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device=device)
        for k, v in inputs.items()
    }
    
    # Generate text output
    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False  # Greedy decoding for deterministic evaluation
        )
        
    # Slice prompt tokens to keep only generated tokens
    prompt_length = model_inputs["input_ids"].shape[1]
    generated_tokens = output_ids[0, prompt_length:]
    prediction = processor.decode(generated_tokens, skip_special_tokens=True).strip()
    
    latency = time.perf_counter() - start_t
    return prediction, latency

# Smoke Test on the first sample
test_item = eval_dataset[0]
test_img = test_item[image_col]
test_gt = test_item[text_col]

print("Running smoke test inference...")
test_pred, test_lat = extract_sinhala_text(test_img, model, processor, DEVICE, DTYPE, config.max_new_tokens)

print("\n--- Smoke Test Results ---")
print(f"Ground Truth: {test_gt}")
print(f"Prediction:   {test_pred}")
print(f"Inference Latency: {test_lat * 1000:.1f} ms")

## 6. Evaluation Metrics: CER, WER & Sinhala Normalization

Sinhala script utilizes combining diacritics (kombuva, al-lakuna, etc.). To prevent artificial mismatch due to Unicode representation:
- We apply **Unicode NFC Normalization** (`unicodedata.normalize('NFC', ...)`).
- We compute **Character Error Rate (CER)** and **Word Error Rate (WER)** via `jiwer`.
- We compute **Exact Match** (1.0 if normalized strings match identically, else 0.0).

In [ ]:
def normalize_sinhala_text(text: str) -> str:
    """
    Normalizes Sinhala text using Unicode NFC and collapses extraneous whitespace.
    """
    if not text:
        return ""
    # Unicode NFC normalization ensures combining diacritics are ordered canonically
    text = unicodedata.normalize("NFC", text)
    # Normalize internal whitespace and strip edges
    text = " ".join(text.split())
    return text

def compute_ocr_metrics(reference: str, hypothesis: str) -> Dict[str, float]:
    """
    Calculates CER, WER, and Exact Match between ground truth and hypothesis.
    """
    norm_ref = normalize_sinhala_text(reference)
    norm_hyp = normalize_sinhala_text(hypothesis)
    
    exact_match = 1.0 if norm_ref == norm_hyp else 0.0
    
    # Handle edge case: empty reference
    if len(norm_ref) == 0:
        cer = 0.0 if len(norm_hyp) == 0 else 1.0
        wer = 0.0 if len(norm_hyp) == 0 else 1.0
        return {"cer": cer, "wer": wer, "exact_match": exact_match}
        
    # Character Error Rate
    cer = jiwer.cer(norm_ref, norm_hyp)
    
    # Word Error Rate
    # If reference has no words (only punctuation), fallback to CER
    if len(norm_ref.split()) == 0:
        wer = cer
    else:
        wer = jiwer.wer(norm_ref, norm_hyp)
        
    return {
        "cer": float(cer),
        "wer": float(wer),
        "exact_match": float(exact_match)
    }

## 7. Run Benchmark Evaluation Loop
Evaluate the dataset across all selected samples, logging per-sample predictions, latency, and error metrics.

In [ ]:
records: List[Dict[str, Any]] = []
print(f"Starting benchmark evaluation on {len(eval_dataset)} samples...")

for idx in tqdm(range(len(eval_dataset)), desc="Evaluating LightOnOCR"): 
    item = eval_dataset[idx]
    img = item[image_col]
    gt_text = str(item[text_col]) if item[text_col] is not None else ""
    
    try:
        pred_text, latency = extract_sinhala_text(
            image=img,
            model=model,
            processor=processor,
            device=DEVICE,
            dtype=DTYPE,
            max_new_tokens=config.max_new_tokens
        )
        
        metrics = compute_ocr_metrics(reference=gt_text, hypothesis=pred_text)
        
        records.append({
            "sample_id": idx,
            "ground_truth": gt_text,
            "predicted_text": pred_text,
            "gt_char_len": len(normalize_sinhala_text(gt_text)),
            "pred_char_len": len(normalize_sinhala_text(pred_text)),
            "gt_word_len": len(normalize_sinhala_text(gt_text).split()),
            "cer": metrics["cer"],
            "wer": metrics["wer"],
            "exact_match": metrics["exact_match"],
            "latency_sec": latency,
            "error": None
        })
    except Exception as e:
        records.append({
            "sample_id": idx,
            "ground_truth": gt_text,
            "predicted_text": "",
            "gt_char_len": len(gt_text),
            "pred_char_len": 0,
            "gt_word_len": len(gt_text.split()),
            "cer": 1.0,
            "wer": 1.0,
            "exact_match": 0.0,
            "latency_sec": 0.0,
            "error": str(e)
        })

df_results = pd.DataFrame(records)
print(f"✅ Evaluation complete. Evaluated {len(df_results)} samples.")

## 8. Benchmark Results Summary
Compute aggregate corpus-level CER, WER, Exact Match %, and latency.

In [ ]:
# Overall corpus metrics
mean_cer = df_results["cer"].mean() * 100
median_cer = df_results["cer"].median() * 100
mean_wer = df_results["wer"].mean() * 100
median_wer = df_results["wer"].median() * 100
exact_match_rate = df_results["exact_match"].mean() * 100
avg_latency_ms = df_results["latency_sec"].mean() * 1000

# Calculate corpus-level CER across concatenated strings for strict academic benchmark
all_refs = [normalize_sinhala_text(r) for r in df_results["ground_truth"]]
all_preds = [normalize_sinhala_text(p) for p in df_results["predicted_text"]]
corpus_cer = jiwer.cer(all_refs, all_preds) * 100
corpus_wer = jiwer.wer(all_refs, all_preds) * 100

summary_data = {
    "Metric": [
        "Corpus CER (%)",
        "Mean Sample CER (%)",
        "Median Sample CER (%)",
        "Corpus WER (%)",
        "Mean Sample WER (%)",
        "Median Sample WER (%)",
        "Exact Match Accuracy (%)",
        "Avg Latency per Sample (ms)",
        "Total Samples Evaluated"
    ],
    "Value": [
        f"{corpus_cer:.2f}%",
        f"{mean_cer:.2f}%",
        f"{median_cer:.2f}%",
        f"{corpus_wer:.2f}%",
        f"{mean_wer:.2f}%",
        f"{median_wer:.2f}%",
        f"{exact_match_rate:.2f}%",
        f"{avg_latency_ms:.1f} ms",
        f"{len(df_results)}"
    ]
}

df_summary = pd.DataFrame(summary_data)
print("=" * 50)
print(f"   BASE ACCURACY BENCHMARK: {config.model_id}")
print("=" * 50)
print(df_summary.to_string(index=False))
print("=" * 50)

# Save results to CSV
df_results.to_csv(config.results_csv_path, index=False, encoding="utf-8-sig")
print(f"💾 Detailed per-sample results saved to: {config.results_csv_path}")

## 9. Visualizations: Error Distributions & Latency
Visualizing the distribution of CER and WER across samples.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. CER Distribution
sns.histplot(df_results["cer"] * 100, bins=25, kde=True, ax=axes[0], color="royalblue")
axes[0].axvline(mean_cer, color="red", linestyle="--", label=f"Mean CER: {mean_cer:.1f}%")
axes[0].set_title("Character Error Rate (CER) Distribution", fontsize=12)
axes[0].set_xlabel("CER (%)")
axes[0].set_ylabel("Count")
axes[0].legend()

# 2. WER Distribution
sns.histplot(df_results["wer"] * 100, bins=25, kde=True, ax=axes[1], color="coral")
axes[1].axvline(mean_wer, color="darkred", linestyle="--", label=f"Mean WER: {mean_wer:.1f}%")
axes[1].set_title("Word Error Rate (WER) Distribution", fontsize=12)
axes[1].set_xlabel("WER (%)")
axes[1].set_ylabel("Count")
axes[1].legend()

# 3. CER vs Text Length
sns.scatterplot(data=df_results, x="gt_char_len", y=df_results["cer"] * 100, ax=axes[2], alpha=0.6, color="purple")
axes[2].set_title("CER vs Ground Truth Character Length", fontsize=12)
axes[2].set_xlabel("Character Count")
axes[2].set_ylabel("CER (%)")

plt.tight_layout()
plt.show()

## 10. Qualitative Analysis: Best vs Worst Predictions
Inspect the model's highest and lowest performing outputs side-by-side.

In [ ]:
def display_case_gallery(sample_indices: List[int], title: str):
    print(f"\n{'=' * 20} {title} {'=' * 20}")
    for idx in sample_indices:
        row = df_results[df_results["sample_id"] == idx].iloc[0]
        img = eval_dataset[idx][image_col].convert("RGB")
        
        plt.figure(figsize=(10, 2.5))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Sample #{idx} | CER: {row['cer']*100:.1f}% | WER: {row['wer']*100:.1f}% | Exact Match: {bool(row['exact_match'])}")
        plt.show()
        
        print(f"[📝 Ground Truth]: {row['ground_truth']}")
        print(f"[🤖 Model Pred  ]: {row['predicted_text']}")
        print("-" * 80)

# Top 3 Best predictions (Lowest CER)
best_indices = df_results.sort_values("cer", ascending=True)["sample_id"].head(3).tolist()
display_case_gallery(best_indices, "TOP 3 BEST PREDICTIONS (Lowest CER)")

# Top 3 Worst predictions (Highest CER)
worst_indices = df_results.sort_values("cer", ascending=False)["sample_id"].head(3).tolist()
display_case_gallery(worst_indices, "TOP 3 WORST PREDICTIONS (Highest CER / Failure Modes)")

## 11. Conclusions & Next Steps for Fine-Tuning

### Key Takeaways from Base Model Benchmark:
1. **Baseline Accuracy**: Records the out-of-the-box CER and WER for `LightOnOCR-2-1B` on Sinhala script.
2. **Script Alignment**: Reveals whether the base model correctly recognizes Sinhala glyphs and diacritics or whether it requires script adaptation.
3. **Target Benchmark**: In prior work on legislative documents, a fine-tuned LightOnOCR-2-1B achieved **1.05% CER** and **5.63% WER**.

### Next Steps in the Project Roadmap:
- **Step 1 (Fine-Tuning)**: Train a LoRA / QLoRA adapter on `Ransaka/sinhala_synthetic_ocr-large` using `LightOnOCR-2-1B-base`.
- **Step 2 (Real-World Evaluation)**: Collect phone camera photos (signs, labels, menus) to benchmark real-world degradation (blur, lighting, angles).
- **Step 3 (TTS Integration)**: Connect OCR output to a Sinhala TTS engine (e.g. Meta MMS Sinhala) to complete the accessibility pipeline.